# Clickbait Classification using DistilBERT and ByT5 Hybrid Model

This notebook trains a character-robust clickbait classification model using DistilBERT semantic representations and ByT5 character-level representations.

In [ ]:
import random
from datasets import load_dataset, Dataset

ds = load_dataset("christinacdl/clickbait_detection_dataset")

mapping = {
    "a": ["@", "4"],
    "e": ["3", "€"],
    "i": ["1", "!"],
    "o": ["0"],
    "s": ["5", "$"],
    "t": ["7"]
}

def modify_text(text, prob=0.30):
    chars = list(text)
    for i, ch in enumerate(chars):
        low = ch.lower()
        if low in mapping and random.random() < prob:
            chars[i] = random.choice(mapping[low])
    return "".join(chars)

augmented = []
for sample in ds["train"]:
    augmented.append(sample)
    if random.random() < 0.30:
        augmented.append({
            "text": modify_text(sample["text"]),
            "label": sample["label"],
            "text_label": sample["text_label"]
        })

train_aug = Dataset.from_list(augmented)
print("Original:", len(ds["train"]))
print("Augmented:", len(train_aug))

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    ByT5Tokenizer,
    T5EncoderModel
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Semantic model
distil_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
distil_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

# Character model
byt_tok = ByT5Tokenizer.from_pretrained("google/byt5-small")
byt_model = T5EncoderModel.from_pretrained("google/byt5-small").to(device)

distil_model.eval()
byt_model.eval()

In [ ]:
from tqdm import tqdm
import torch

BATCH_SIZE = 64
X = []
Y = []

for start in tqdm(range(0, len(train_aug), BATCH_SIZE)):
    batch = train_aug[start:start+BATCH_SIZE]
    texts = batch["text"]
    labels = batch["label"]

    with torch.no_grad():
        d_inputs = distil_tok(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        d_vec = distil_model(**d_inputs).last_hidden_state[:, 0, :]

        b_inputs = byt_tok(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        b_vec = byt_model(**b_inputs).last_hidden_state.mean(dim=1)

        fused = torch.cat([d_vec, b_vec], dim=1)

    X.append(fused.cpu())
    Y.extend(labels)

X = torch.cat(X).float()
Y = torch.tensor(Y).long()
print("Feature tensor shape:", X.shape)
print("Label tensor shape:", Y.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

train_dataset = TensorDataset(X, Y)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

class FusionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2240, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        self.out = nn.Linear(128, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)
        return self.out(x)

fusion_model = FusionClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=0.001)

EPOCHS = 5
for epoch in range(EPOCHS):
    fusion_model.train()
    total_loss, correct, total = 0, 0, 0
    for xb, yb in tqdm(train_loader):
        xb, yb = xb.to(device), yb.to(device)
        outputs = fusion_model(xb)
        loss = criterion(outputs, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    print(f"Epoch {epoch+1}: Loss = {round(total_loss, 3)}, Accuracy = {round(100*correct/total, 2)}%")

torch.save(fusion_model.state_dict(), "fusion_model.pth")
print("Model saved to fusion_model.pth")

In [ ]:
fusion_model.eval()
labels = {0: "Non-Clickbait", 1: "Clickbait"}

def predict(headline):
    d = distil_tok(headline, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    b = byt_tok(headline, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        semantic = distil_model(**d).last_hidden_state[:,0,:]
        character = byt_model(**b).last_hidden_state.mean(dim=1)
        fused = torch.cat([semantic, character], dim=1)
        logits = fusion_model(fused)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1).item()
    print("Headline   :", headline)
    print("Prediction :", labels[pred])
    print("Confidence :", round(probs[0][pred].item()*100, 2), "%")

predict("Y0u W0N'T BEL1EVE TH1S!!!")
predict("Sc13ntists D1sc0ver N3w Pl@n3t")